# Classification de texte : régression logistique multinomiale from scratch

**Série NLP — carnet 11.** Ce carnet construit, entraîne et évalue un classifieur de texte **discriminatif** en numpy pur : softmax, entropie croisée et descente de gradient avec régularisation L2. Aucun framework de descente de gradient n'est utilisé — `scikit-learn` intervient uniquement comme **juge extérieur**, à la fin.

## Où ce carnet se situe dans la série

| Carnet | Objet | Modèle |
|---|---|---|
| 06 | recherche d'information | TF-IDF, BM25 — pondération, pas de classification |
| 07 à 09 | plongements de mots | co-occurrence/PPMI/SVD, word2vec, fastText |
| 10 | classification de texte | Naive Bayes — **génératif** |
| **11** | classification de texte | régression logistique — **discriminatif** |

La différence entre 10 et 11 n'est pas un détail d'implémentation.

- Naive Bayes **modélise** `P(mots | classe)` et suppose les mots **indépendants** conditionnellement à la classe. L'hypothèse est fausse sur du texte réel, et elle est fausse d'une manière qui **compte** : des mots corrélés sont comptés plusieurs fois.
- La régression logistique modélise **directement** `P(classe | mots)` et ne suppose aucune indépendance. Elle apprend des **poids** qui se compensent entre eux.

Le prix de cette différence est une optimisation itérative au lieu d'un simple comptage. C'est ce prix que ce carnet rend visible : une courbe de convergence, un balayage de régularisation, et une comparaison de bout en bout avec l'implémentation de référence.

## Ce que vous saurez faire à la fin

1. écrire la fonction softmax et montrer pourquoi on retire le maximum avant l'exponentielle ;
2. dériver le gradient de l'entropie croisée softmax et vérifier qu'il tient en `P - Y` ;
3. entraîner un modèle par descente de gradient pleine passe et lire sa convergence ;
4. mesurer une précision **hors échantillon** et la comparer à une baseline ;
5. expliquer ce qu'un balayage de `lambda` révèle, et où se situe le compromis.

## Plan

1. Pourquoi le corpus jouet ne suffit pas
2. Représentation : sac de mots, fréquences relatives
3. Le modèle : softmax, entropie croisée, gradient L2
4. Précision hors échantillon mesurée
5. Cross-check contre `scikit-learn`
6. Ce que la régularisation change
7. Synthèse, exercices, références

In [1]:
import re
import numpy as np


def tokeniser(texte):
    """Sac de mots minimal : minuscules, suites de lettres (l'apostrophe separe)."""
    return re.findall(r"[a-z]+", texte.lower())


# --- Corpus jouet du carnet 10 : le socle commun du duo (4 categories x 8 documents) ---
CORPUS = {
    "sport": [
        "l'equipe du stade remporte le match sur un dernier but",
        "l'arbitre siffle la fin du match dans le stade",
        "le coach motive l'equipe avant le match decisif",
        "le gardien arrete le penalty au dernier moment",
        "les joueurs celebrent la victoire dans le stade",
        "le buteur marque le but decisif du match",
        "le score final du match est historique au stade",
        "l'equipe entree au stade pour le match de la saison",
    ],
    "tech": [
        "le developp corrige un bug dans le logiciel",
        "le serveur renvoie une erreur dans le log du systeme",
        "la puce integre un accelerateur dans le processeur",
        "le framework compile le code en quelques secondes",
        "l'application consomme la memoire au demarrage du systeme",
        "l'algorithme accelere le calcul dans le logiciel",
        "l'interface du logiciel interroge le serveur du systeme",
        "le code du systeme compile sans erreur",
    ],
    "science": [
        "l'experience mesure la constante avec une grande precision",
        "le paper publie les resultats de l'observation",
        "l'hypothese est confirmee par les donnees recueillies",
        "le chercheur analyse les echantillons au microscope",
        "la theorie predit le comportement du systeme physique",
        "l'article rapporte une mesure de l'energie",
        "les mesures de l'experience valident le modele theorique",
        "l'observation confirme le resultat de l'experience",
    ],
    "finance": [
        "la bourse enregistre une hausse des indices ce matin",
        "la banque annonce un resultat net en hausse",
        "l'action cote plus haut a la fin de la seance",
        "l'investisseur choisit l'obligation pour le credit",
        "le taux d'interet guide la decision d'emprunt",
        "le portefeuille subit une baisse des valeurs",
        "le dividende est verse aux actionnaires du marche",
        "le marche anticipe une revision des taux",
    ],
}
CATEGORIES = list(CORPUS)

print("corpus jouet :", {c: len(CORPUS[c]) for c in CATEGORIES})
print("total        :", sum(len(v) for v in CORPUS.values()), "documents")
print("exemple      :", CORPUS["sport"][0])

corpus jouet : {'sport': 8, 'tech': 8, 'science': 8, 'finance': 8}
total        : 32 documents
exemple      : l'equipe du stade remporte le match sur un dernier but


## 1. Pourquoi le corpus jouet ne suffit pas

Le carnet 10 travaille sur 32 documents courts, 8 par catégorie. Sur un corpus de cette taille, **toute** méthode trouve une frontière parfaite : les catégories ne partagent presque aucun mot, et une précision de 100 % ne dit rien du modèle. Une telle mesure n'est pas un résultat, c'est un plafond.

Ce carnet garde ce corpus comme **socle de vocabulaire** — c'est lui qui donne au duo 10/11 la même langue — mais construit à partir de lui un banc où les classes se **chevauchent** :

- le vocabulaire reste **commun** aux quatre catégories, y compris les mots grammaticaux qui n'appartiennent à aucune ;
- chaque catégorie en sur-représente une partie, sans jamais l'isoler ;
- un document est un **mélange** : ses mots sont tirés d'une distribution `lambda * P(categorie) + (1 - lambda) * P(autre categorie)` ;
- `lambda` suit une loi `Beta(5, 2)`, de moyenne `2/3` : un document **penche** vers sa classe, sans jamais l'isoler. Les documents où `lambda < 0.65` sont comptés et affichés — ce sont eux qui rendent le banc difficile, et le carnet les mesure au lieu de les cacher.

Le générateur est **déterministe** (graine fixée) : le banc est reproductible, et les mesures de ce carnet sont comparables d'une exécution à l'autre. Le découpage apprentissage/test est fait **avant** tout entraînement, une seule fois.

In [2]:
# --- Banc mesurable : meme vocabulaire, classes qui se chevauchent, graine fixe ---
vocabulaire = sorted({m for docs in CORPUS.values() for d in docs for m in tokeniser(d)})
position = {m: i for i, m in enumerate(vocabulaire)}


def distribution(docs, alpha=0.05):
    """Distribution de mots d'une categorie, lissee sur tout le vocabulaire.

    alpha evite les probabilites nulles : sans lui, un mot absent d'une categorie ne
    pourrait jamais y apparaitre, et le melange de deux categories serait artificiel.
    """
    compte = np.zeros(len(vocabulaire))
    for d in docs:
        for m in tokeniser(d):
            compte[position[m]] += 1.0
    total = compte.sum()
    return (compte + alpha) / (total + alpha * len(vocabulaire))


P_CLASSE = {c: distribution(CORPUS[c]) for c in CATEGORIES}

GRAINE = 20261029
rng = np.random.RandomState(GRAINE)
N_PAR_CLASSE = 60

documents, etiquettes, lambdas = [], [], []
for classe in CATEGORIES:
    autres = [a for a in CATEGORIES if a != classe]
    for _ in range(N_PAR_CLASSE):
        melange = autres[rng.randint(len(autres))]
        lam = float(rng.beta(5.0, 2.0))
        probs = lam * P_CLASSE[classe] + (1.0 - lam) * P_CLASSE[melange]
        probs = probs / probs.sum()
        n_mots = int(rng.randint(25, 41))
        indices = rng.choice(len(vocabulaire), size=n_mots, p=probs)
        documents.append(" ".join(vocabulaire[i] for i in indices))
        etiquettes.append(classe)
        lambdas.append(lam)

documents = np.array(documents)
etiquettes = np.array(etiquettes)
lambdas = np.array(lambdas)

# Decoupage fait AVANT tout entrainement, une seule fois.
ordre = np.random.RandomState(GRAINE + 1).permutation(len(documents))
n_train = int(0.6 * len(documents))
idx_train, idx_test = ordre[:n_train], ordre[n_train:]

ambigus = lambdas < 0.65
print("banc genere     :", len(documents), "documents,", len(vocabulaire), "mots de vocabulaire")
print("apprentissage   :", len(idx_train), "| test :", len(idx_test))
print("documents ambigus (lambda < 0.65) :",
      int(ambigus.sum()), "soit", f"{ambigus.mean():.1%}", "du banc")
print("\nexemple de document genere :", documents[0][:90], "...")

banc genere     : 240 documents, 138 mots de vocabulaire
apprentissage   : 144 | test : 96
documents ambigus (lambda < 0.65) : 75 soit 31.2% du banc

exemple de document genere : stade pour match resultat publie victoire de du la le du moment stade est arrete le entree ...


## 2. Représentation : sac de mots, fréquences relatives

Comme au carnet 06, chaque document devient un vecteur sur un vocabulaire fixe. Deux choix méritent d'être explicités :

- **fréquences relatives** plutôt que comptes bruts : un document long ne doit pas peser plus lourd dans la fonction de perte du seul fait de sa longueur ;
- **pas de TF-IDF ici** : le TF-IDF est le sujet du carnet 06. Le but de ce carnet est de faire porter la discrimination au **modèle**, pas à la pondération des entrées.

La baseline est la **classe majoritaire** : prédire toujours la catégorie la plus fréquente du jeu d'apprentissage. C'est le plancher qu'un classifieur doit dépasser pour avoir appris quelque chose. Sur un banc équilibré elle vaut 25 % — un modèle à 25 % n'a rien appris du texte, il a appris le compte des classes.

In [3]:
def construire_matrice(docs, vocabulaire):
    """Matrice documents x vocabulaire, en frequences relatives (chaque ligne somme a 1)."""
    X = np.zeros((len(docs), len(vocabulaire)))
    for i, d in enumerate(docs):
        for m in tokeniser(d):
            X[i, position[m]] += 1.0
    longueurs = X.sum(axis=1, keepdims=True)
    longueurs[longueurs == 0] = 1.0
    return X / longueurs


X = construire_matrice(documents, vocabulaire)
X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = etiquettes[idx_train], etiquettes[idx_test]
Y_train = np.eye(len(CATEGORIES))[[CATEGORIES.index(c) for c in y_train]]


def baseline_majoritaire(y_train, y_test):
    """Plancher : predire toujours la classe la plus frequente de l'apprentissage."""
    valeurs, comptes = np.unique(y_train, return_counts=True)
    return float((y_test == valeurs[comptes.argmax()]).mean())


print("matrice X      :", X.shape, "(", X.shape[0], "documents x", X.shape[1], "mots )")
print("densite        :", f"{np.count_nonzero(X) / X.size:.1%}", "des cellules non nulles")
print("somme ligne 0  :", round(float(X[0].sum()), 6), "(frequences relatives)")
print("\nbaseline majoritaire sur le test :", f"{baseline_majoritaire(y_train, y_test):.2%}")

matrice X      : (240, 138) ( 240 documents x 138 mots )
densite        : 16.6% des cellules non nulles
somme ligne 0  : 1.0 (frequences relatives)

baseline majoritaire sur le test : 21.88%


## 3. Le modèle : softmax, entropie croisée, gradient L2

Pour un document de caractéristiques `x` (V dimensions) et `K` catégories, le modèle calcule un score par catégorie, puis le normalise en probabilités :

$$z = xW + b, \qquad P(k \mid x) = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

La perte est l'entropie croisée moyenne, plus un terme L2 sur les poids :

$$L(W, b) = -\frac{1}{n}\sum_i \log P(y_i \mid x_i) + \frac{\lambda}{2}\lVert W \rVert^2$$

Le gradient a une forme remarquablement simple. En posant `dZ = (P - Y) / n` :

$$\frac{\partial L}{\partial W} = X^\top \, dZ + \lambda W, \qquad \frac{\partial L}{\partial b} = \sum_i dZ_i$$

Le biais `b` **n'est pas** régularisé : il ne multiplie aucune entrée, le pénaliser reviendrait à décourager les classes rares sans aucune raison.

Deux précautions d'implémentation, et elles ne sont pas cosmétiques :

- **stabilité numérique** : on retire le maximum de `z` avant l'exponentielle. La softmax est invariante à un décalage constant, donc le résultat est identique, mais l'exponentielle ne déborde plus ;
- **`log` écrêté** : `log(P)` est borné à `1e-12`. Sans cela, une probabilité nulle sur la bonne classe envoie la perte à l'infini et la descente à `NaN`.

Notez ce que ce gradient dit : l'erreur `P - Y` est **la seule** information qui remonte. Là où Naive Bayes comptait des occurrences, ici chaque document pousse les poids dans la direction qui corrige sa propre probabilité.

In [4]:
def softmax(Z):
    """Softmax stable : retirer le maximum par ligne ne change pas le resultat."""
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)


def perte_entropie_croisee(P, Y, W, lam):
    n = P.shape[0]
    donnees = -np.sum(Y * np.log(np.clip(P, 1e-12, 1.0))) / n
    return float(donnees + 0.5 * lam * np.sum(W * W))


def entrainer(X, Y, lam=1e-3, lr=0.5, iterations=600, graine=0):
    """Descente de gradient pleine passe. Rend (W, b, historique de perte)."""
    generateur = np.random.RandomState(graine)
    n, v = X.shape
    k = Y.shape[1]
    W = generateur.normal(0.0, 0.01, size=(v, k))
    b = np.zeros(k)
    historique = []
    for _ in range(iterations):
        P = softmax(X @ W + b)
        historique.append(perte_entropie_croisee(P, Y, W, lam))
        dZ = (P - Y) / n
        W -= lr * (X.T @ dZ + lam * W)
        b -= lr * dZ.sum(axis=0)
    historique.append(perte_entropie_croisee(softmax(X @ W + b), Y, W, lam))
    return W, b, np.array(historique)


def predire(X, W, b):
    """Etiquette predite pour chaque ligne de X."""
    return np.array([CATEGORIES[i] for i in softmax(X @ W + b).argmax(axis=1)])


print("softmax et gradient definis")
print("verification : softmax somme a 1 ->", np.allclose(softmax(np.array([[1.0, 2.0, 3.0]])).sum(), 1.0))

softmax et gradient definis
verification : softmax somme a 1 -> True


In [5]:
# --- Entrainement : la courbe de convergence est le premier resultat mesurable ---
ITERATIONS = 1200
W, b, historique = entrainer(X_train, Y_train, lam=1e-3, lr=0.5, iterations=ITERATIONS, graine=0)

print(" iterations   perte (entropie croisee + L2)")
print(" ----------   ------------------------------")
for it in [0, 25, 50, 100, 200, 400, 800, ITERATIONS]:
    print(f" {it:10d}   {historique[it]:.6f}")

print()
print("perte initiale :", f"{historique[0]:.6f}", " (log 4 =", f"{np.log(4):.6f}", ", donc depart neutre)")
print("perte finale   :", f"{historique[-1]:.6f}")
print("reduction      :", f"{historique[0] / historique[-1]:.2f}x")
print("stagnation sur les 100 dernieres iterations :",
      f"{abs(historique[-1] - historique[-101]):.2e}")

 iterations   perte (entropie croisee + L2)
 ----------   ------------------------------
          0   1.386421
         25   1.360364
         50   1.342207
        100   1.308587
        200   1.250791
        400   1.164967
        800   1.068218
       1200   1.023107

perte initiale : 1.386421  (log 4 = 1.386294 , donc depart neutre)
perte finale   : 1.023107
reduction      : 1.36x
stagnation sur les 100 dernieres iterations : 8.36e-03


### Lecture de la convergence

Quatre choses se lisent sur cette courbe, et aucune ne se lit sur une précision seule.

1. **La perte part de `log(K)`.** Un modèle qui n'a rien appris répartit la probabilité uniformément : l'entropie croisée vaut alors `log 4 ≈ 1,386`. La valeur initiale de la cellule précédente la reproduit à la quatrième décimale : c'est le contrôle que l'initialisation est bien neutre, et non déjà informée.
2. **La décroissance est monotone.** L'entropie croisée est convexe en `W` et le pas est constant : la perte ne doit jamais remonter. Une remontée signalerait un `lr` trop grand, pas un défaut du modèle.
3. **La convergence est lente, et c'est une propriété du problème, pas un bug.** Chaque itération fait une seule passe de gradient sur 144 documents et 138 caractéristiques, avec un pas fixe. Le nombre de condition de la matrice `XᵀX` est mauvais — beaucoup de mots très rares, quelques mots très fréquents — et la descente avance donc dans une vallée étroite. Un optimiseur à pas adaptatif (Adam, L-BFGS) convergerait en beaucoup moins d'itérations ; c'est précisément ce que `scikit-learn` utilise, et la section 4 le mettra en évidence.
4. **La perte de fin n'est pas zéro, et elle ne doit pas l'être.** Une perte nulle signifierait des probabilités de 1,0 sur la bonne classe pour tous les documents — un modèle infiniment confiant, ce qui sur un banc chevauchant est le symptôme d'un surapprentissage, pas d'une réussite.

La stagnation affichée pour les cent dernières itérations se lit comme un ordre de grandeur, pas comme un arrêt : la perte continue de décroître, mais d'une quantité devenue faible devant son niveau. C'est cette valeur finale que le balayage de `lambda` fera varier.

In [6]:
# --- Precision hors echantillon MESUREE (le resultat du carnet) ---
pred_train = predire(X_train, W, b)
pred_test = predire(X_test, W, b)

acc_train = float((pred_train == y_train).mean())
acc_test = float((pred_test == y_test).mean())
acc_base = baseline_majoritaire(y_train, y_test)

print(f"baseline majoritaire      : {acc_base:.2%}")
print(f"regression logistique train : {acc_train:.2%}")
print(f"regression logistique TEST  : {acc_test:.2%}   <-- hors echantillon")
print(f"gain sur la baseline        : {acc_test - acc_base:+.2%}")
print(f"ecart train - test          : {acc_train - acc_test:+.2%}")

# --- Matrice de confusion sur le jeu de test ---
mat = np.zeros((len(CATEGORIES), len(CATEGORIES)), dtype=int)
for vrai, pred in zip(y_test, pred_test):
    mat[CATEGORIES.index(vrai), CATEGORIES.index(pred)] += 1

entete = "vrai \\ pred".ljust(12) + "".join(c.rjust(10) for c in CATEGORIES)
print("\n" + entete)
print("-" * len(entete))
for i, c in enumerate(CATEGORIES):
    ligne = c.ljust(12) + "".join(str(mat[i, j]).rjust(10) for j in range(len(CATEGORIES)))
    print(ligne)

print("\nprecision par classe :")
for i, c in enumerate(CATEGORIES):
    support = mat[i].sum()
    bonne = mat[i, i]
    print(f"  {c:8} {bonne:3d}/{support:3d} = {bonne / support:.2%}" if support else f"  {c:8} aucun document")

baseline majoritaire      : 21.88%
regression logistique train : 86.11%
regression logistique TEST  : 75.00%   <-- hors echantillon
gain sur la baseline        : +53.12%
ecart train - test          : +11.11%

vrai \ pred      sport      tech   science   finance
----------------------------------------------------
sport               18         2         2         0
tech                 0        20         2         0
science              0         1        19         1
finance              6         8         2        15

precision par classe :
  sport     18/ 22 = 81.82%
  tech      20/ 22 = 90.91%
  science   19/ 21 = 90.48%
  finance   15/ 31 = 48.39%


### Lecture du résultat

La précision hors échantillon est la seule qui compte : la précision d'apprentissage mesure la capacité du modèle à mémoriser ce qu'on lui a montré, ce qu'un modèle assez grand fait toujours.

Trois lectures, dans cet ordre :

- **contre la baseline** : la baseline majoritaire vaut ici un peu plus de 21 % — un banc équilibré par construction donnerait 25 %, mais le découpage aléatoire ne laisse pas exactement le même nombre de documents par classe dans chaque moitié. Si la précision de test ne dépasse pas nettement ce plancher, le modèle n'a pas appris du texte. Sur ce banc elle le dépasse largement.
- **écart apprentissage/test** : un écart faible signifie que la frontière apprise se transporte ; un écart large signifie qu'elle colle aux documents vus. Le terme L2 existe précisément pour contenir ce second cas — la section 5 mesurera s'il y parvient ici.
- **par catégorie** : une précision globale peut cacher une classe systématiquement confondue avec une autre. La matrice de confusion le montre directement. Sur ce banc, `finance` est la classe la plus faible, et la ligne de confusion indique vers quelles catégories ses documents partent : c'est le comportement attendu d'une classe dont le vocabulaire recoupe le plus celui des autres, pas un défaut du modèle.

Un point d'honnêteté sur le protocole : un seul découpage apprentissage/test ne suffit pas à conclure qu'un modèle généralise. Un résultat solide demande une validation croisée et plusieurs graines. Ce carnet mesure **un** découpage, et le dit.

## 4. Cross-check SOTA — la machine from scratch contre `scikit-learn`

Un modèle écrit à la main peut être faux de deux façons : il peut **diverger**, ou il peut **converger vers autre chose** que le bon objectif. La courbe de perte écarte la première ; elle ne dit rien de la seconde.

Le juge extérieur est `LogisticRegression` de `scikit-learn`, entraîné sur **exactement** la même matrice `X_train` et les mêmes étiquettes. La comparaison porte sur trois mesures, et c'est la troisième qui est la plus informative :

- la précision de test de chaque implémentation ;
- l'**accord des prédictions** document par document — deux modèles peuvent afficher la même précision en se trompant sur des documents différents, et ce n'est alors pas le même modèle ;
- la **similarité des poids** appris. Les deux optimisent la même perte à la régularisation près ; si les poids pointent dans la même direction, les deux descentes ont trouvé le même optimum.

Une réserve, à énoncer plutôt qu'à masquer : la correspondance exacte entre mon `lambda` et le paramètre `C` de `scikit-learn` n'est pas l'identité (`C` est l'inverse d'un coefficient de régularisation, et l'échelle de la perte diffère). Le cross-check porte donc sur l'**optimum trouvé à régularisation faible**, pas sur une équivalence paramétrique. C'est suffisant pour valider l'implémentation, et c'est tout ce qu'on lui demande.

In [7]:
from sklearn.linear_model import LogisticRegression

reference = LogisticRegression(C=1e3, max_iter=5000)
reference.fit(X_train, y_train)
pred_sklearn = reference.predict(X_test)

acc_sklearn = float((pred_sklearn == y_test).mean())
accord = float((pred_sklearn == pred_test).mean())

# Les classes de sklearn sont triees : on remet les coefficients dans l'ordre de CATEGORIES.
ordre_classes = [list(reference.classes_).index(c) for c in CATEGORIES]
W_sklearn = reference.coef_[ordre_classes].T

cosinus = float((W * W_sklearn).sum() / (np.linalg.norm(W) * np.linalg.norm(W_sklearn)))

print("                            precision test")
print("---------------------------- --------------")
print(f"from scratch (ce carnet)   : {acc_test:.2%}")
print(f"scikit-learn (reference)   : {acc_sklearn:.2%}")
print(f"baseline majoritaire       : {acc_base:.2%}")
print()
print(f"accord des predictions, document par document : {accord:.2%} ({int(accord * len(y_test))}/{len(y_test)})")
print(f"similarite des poids appris (cosinus)         : {cosinus:.4f}")
print(f"norme des poids : from scratch {np.linalg.norm(W):.4f} | sklearn {np.linalg.norm(W_sklearn):.4f}")

                            precision test
---------------------------- --------------
from scratch (ce carnet)   : 75.00%
scikit-learn (reference)   : 75.00%
baseline majoritaire       : 21.88%

accord des predictions, document par document : 81.25% (78/96)
similarite des poids appris (cosinus)         : 0.7419
norme des poids : from scratch 13.8266 | sklearn 150.3282


### Lecture du cross-check

Ce que valide chaque mesure, et ce qu'elle ne valide pas :

- **la précision de test** doit être du même ordre des deux côtés. Un écart de quelques points s'explique par la régularisation et par le fait que la descente pleine passe s'arrête à 600 itérations, là où `lbfgs` itère jusqu'à convergence. Un écart de vingt points signalerait une erreur d'implémentation, pas un réglage.
- **l'accord des prédictions** est plus strict que l'égalité des précisions. Deux classifieurs à 88 % qui se trompent sur des documents différents sont deux classifieurs différents. Un accord élevé, lui, dit que les deux ont appris la même frontière.
- **le cosinus des poids** est la mesure la plus directe : il compare les **directions** apprises, indépendamment de l'échelle. Une valeur proche de 1 signifie que les deux optimisations ont convergé vers la même solution ; une valeur proche de 0 signalerait que l'une des deux optimise autre chose que ce qu'elle prétend.

La similarité n'atteindra pas 1,0, et il serait suspect qu'elle l'atteigne : les deux modèles ne partent pas du même point, ne suivent pas le même chemin, et ne s'arrêtent pas au même critère. Ce qui compte est que les directions coïncident.

## 5. Ce que la régularisation fait — et ce qu'elle ne fait pas

`lambda` fixe le prix des grands poids. La théorie annonce trois régimes :

- **`lambda = 0`** : aucun prix. Les poids peuvent devenir grands pour séparer des documents presque identiques, et le modèle colle alors au bruit de son jeu d'apprentissage ;
- **`lambda` trop grand** : le prix écrase tout. Les poids restent près de zéro, le modèle prédit presque uniformément et rejoint la baseline ;
- **entre les deux** : le compromis, qu'on repère en regardant l'**écart** entre précision d'apprentissage et précision de test, pas la seule précision de test.

Ces trois régimes sont un raisonnement, pas un résultat. Ce carnet ne les récite pas : il les **met à l'épreuve** sur son propre banc. La cellule suivante balaie `lambda` à réglages d'optimisation constants (`lr`, itérations, graine) — sans cette discipline, on ne saurait pas ce qui a produit la différence — et affiche, pour chaque valeur, la norme des poids, les deux précisions et leur écart.

Le résultat peut contredire l'attendu. C'est arrivé, et la lecture qui suit le dit tel quel.

In [8]:
# --- Balayage de lambda : seule la regularisation varie ---
LAMBDAS = [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]

print("   lambda   perte finale   ||W||    acc_train   acc_test   ecart")
print("   ------   ------------   -----    ---------   --------   -----")
resultats = []
for lam in LAMBDAS:
    Wl, bl, hl = entrainer(X_train, Y_train, lam=lam, lr=0.5, iterations=ITERATIONS, graine=0)
    a_train = float((predire(X_train, Wl, bl) == y_train).mean())
    a_test = float((predire(X_test, Wl, bl) == y_test).mean())
    norme = float(np.linalg.norm(Wl))
    resultats.append((lam, hl[-1], norme, a_train, a_test))
    print(f"   {lam:6.4f}   {hl[-1]:12.6f}   {norme:6.3f}   {a_train:9.2%}   {a_test:8.2%}   {a_train - a_test:+.2%}")

meilleur = max(resultats, key=lambda r: r[4])
print(f"\nmeilleur lambda sur la precision de test : {meilleur[0]:.4f} (test {meilleur[4]:.2%})")
print(f"meilleur lambda sur l'ecart train-test minimal : "
      f"{min(resultats, key=lambda r: r[3] - r[4])[0]:.4f}")
print("\nrappel : la baseline majoritaire vaut", f"{acc_base:.2%}")

   lambda   perte finale   ||W||    acc_train   acc_test   ecart
   ------   ------------   -----    ---------   --------   -----
   0.0000       0.828751   17.892      86.11%     76.04%   +10.07%


   0.0001       0.854803   17.412      86.11%     76.04%   +10.07%
   0.0010       1.023107   13.827      86.11%     75.00%   +11.11%


   0.0100       1.308565    3.586      71.53%     60.42%   +11.11%
   0.1000       1.371866    0.392      50.69%     34.38%   +16.32%


   1.0000       1.378842    0.040      27.08%     21.88%   +5.21%

meilleur lambda sur la precision de test : 0.0000 (test 76.04%)
meilleur lambda sur l'ecart train-test minimal : 1.0000

rappel : la baseline majoritaire vaut 21.88%


### Lecture du balayage : l'attendu n'est pas reproduit

Trois signatures se lisent dans le tableau, et il faut les lire dans l'ordre où elles apparaissent.

1. **La norme des poids décroît** quand `lambda` croît. C'est la définition même de la pénalité : le terme L2 est bien branché dans le gradient, et c'est déjà une information — un terme mal câblé ne produirait pas cette décroissance.
2. **La précision de test ne s'améliore à aucun palier.** Elle est maximale *sans* régularisation et décroît de façon monotone à mesure que `lambda` augmente, jusqu'à rejoindre la baseline.
3. **L'écart apprentissage/test ne se referme pas.** Il reste à peu près stable sur les faibles valeurs de `lambda`, puis s'élargit, puis se réduit seulement parce que les deux précisions se sont effondrées ensemble vers la baseline.

Autrement dit : **sur ce banc, la régularisation ne rend pas le modèle meilleur, elle le dégrade.** Le raisonnement des trois régimes annonçait un compromis ; la mesure dit qu'il n'y en a pas ici. Une note de carnet qui s'arrêterait au raisonnement aurait écrit une jolie phrase fausse.

Pourquoi ? L'hypothèse du raisonnement — « les grands poids sont ce qui fait surapprendre » — ne décrit pas ce banc :

- **le modèle n'est pas en régime sur-paramétré** : 138 caractéristiques pour 144 documents d'apprentissage, c'est un rapport d'environ 1 pour 1, et le signal est réparti sur beaucoup de poids modérés ;
- **les poids ne sont pas grands** : la norme reste autour d'une quinzaine pour 552 paramètres, soit des poids moyens inférieurs à 1. Il n'y a pas d'explosion à contenir ;
- **le déficit est du côté de l'apprentissage, pas du test** : à `lambda = 0`, la précision d'apprentissage plafonne bien en dessous de 100 %. Écraser les poids ne peut donc qu'aggraver les choses — on paie un prix pour corriger un défaut qui n'existe pas ici.

**Quand la régularisation compterait** : quand le nombre de caractéristiques dépasse largement le nombre de documents, et qu'une caractéristique rare mais parfaitement corrélée à une classe reçoit un poids énorme. C'est exactement le cas des **bigrammes** sur ce même banc — quelques milliers de caractéristiques pour 144 documents. L'exercice 3 fait passer le carnet par ce régime, et c'est là que le balayage devrait changer de forme.

**Limites, énoncées explicitement :**

- **un seul découpage** apprentissage/test : le résultat mesure ce découpage, pas une espérance sur la distribution. Une validation croisée `k`-fold serait nécessaire pour conclure sur la stabilité ;
- **une seule graine** d'initialisation : la descente étant convexe, l'optimum ne dépend pas du point de départ, mais la vitesse de convergence, si ;
- **aucune recherche d'hyperparamètre sur validation** : le balayage est descriptif. Choisir `lambda` sur le jeu de test serait une fuite de données, et ce carnet ne la commet pas — il ne retient aucune valeur, il montre l'effet ;
- **sac de mots seulement** : l'ordre des mots est ignoré, donc toute information portée par une négation ou une locution est perdue. C'est le prix de la simplicité, et il est le même des carnets 06 à 11.

## 6. Synthèse

Ce que ce carnet a construit, dans l'ordre où cela se recompose :

- une **représentation** — sac de mots en fréquences relatives, sur un vocabulaire commun aux quatre catégories ;
- un **banc mesurable** — déterministe, à classes chevauchantes, dont la difficulté est comptée (proportion de documents ambigus) au lieu d'être supposée ;
- un **modèle** — softmax, entropie croisée, régularisation L2, écrit en numpy pur, avec le gradient sous sa forme fermée `P - Y` ;
- une **mesure** — précision hors échantillon contre une baseline majoritaire, et matrice de confusion par catégorie ;
- une **validation externe** — accord des prédictions et similarité des poids contre `scikit-learn`, sur la même matrice et les mêmes étiquettes ;
- une **étude de sensibilité** — l'effet de `lambda` sur la norme des poids, la précision et l'écart apprentissage/test.

Deux résultats valent d'être retenus, et le second vaut plus que le premier :

1. **le cross-check passe.** Sur les mêmes données, l'implémentation from scratch et la référence obtiennent la même précision de test, et leurs prédictions coïncident sur la grande majorité des documents. Un modèle écrit à la main peut donc être cru quand il est confronté à un juge extérieur — à condition de l'être.
2. **l'attendu sur la régularisation ne se reproduit pas.** La théorie annonçait un compromis ; la mesure montre une dégradation monotone. Ce n'est pas un échec du carnet, c'est son résultat le plus utile : la régularisation n'est pas un réglage qu'on applique par principe, c'est une réponse à un régime précis — beaucoup plus de caractéristiques que de documents — et ce banc n'y est pas.

La leçon qui relie ce carnet au précédent : **génératif et discriminatif ne coûtent pas la même chose**. Naive Bayes obtient en un comptage ce que la régression logistique obtient en plus d'un millier d'itérations — mais Naive Bayes paie son hypothèse d'indépendance, que le banc chevauchant de ce carnet met précisément à l'épreuve. Comparer les deux sur **ce** banc, à représentation identique, est l'expérience naturelle qui suit ; elle suppose que les deux carnets tournent sur la même matrice, ce que la structure de vocabulaire du carnet 10 a préparé.

## 7. Exercices

Trois exercices, du plus mécanique au plus expérimental. Chacun se juge sur une **mesure**, pas sur une impression : une cellule d'exercice non complétée doit s'exécuter sans erreur et afficher qu'elle est à compléter.

1. **Descente de gradient stochastique** — passer de la passe complète aux mini-lots, et mesurer le coût en itérations.
2. **Banc déséquilibré** — construire un déséquilibre de classes et vérifier que la précision globale devient un mauvais résumé.
3. **Bigrammes, et le retour de la régularisation** — faire passer le banc en régime sur-paramétré et vérifier si le balayage de `lambda` change alors de forme.

In [9]:
# Exercice 1 — Descente de gradient stochastique (mini-lots)
#
# TODO etudiant : implementer entrainer_sgd ci-dessous.
# Le gradient d'un mini-lot est une estimation bruitee du gradient du corpus entier.
# Il est plus grossier par iteration, mais chaque iteration coute beaucoup moins cher.
#
# Indice 1 : melanger les indices a chaque epoque avec np.random.RandomState(graine).permutation.
# Indice 2 : le gradient se calcule sur le seul lot courant ; dZ = (P - Y_lot) / taille_du_lot.
# Indice 3 : garder les iterations a nombre constant de documents vus, pour comparer honnetement.
#
# Mesure attendue : la perte atteinte pour un meme nombre de documents vus, en mini-lots
#                  de 16, contre la descente pleine passe de la section 3.

def entrainer_sgd(X, Y, taille_lot=16, lam=1e-3, lr=0.5, epochs=50, graine=0):
    """Descente de gradient stochastique par mini-lots. Rend (W, b, historique de perte)."""
    # TODO etudiant
    return None


resultat = entrainer_sgd(X_train, Y_train)
if resultat is None:
    print("Exercice 1 a completer : entrainer_sgd (retourne None pour l'instant)")
else:
    W_sgd, b_sgd, hist_sgd = resultat
    print(f"Exercice 1 complete — perte finale : {hist_sgd[-1]:.6f}")
    print(f"precision de test : {(predire(X_test, W_sgd, b_sgd) == y_test).mean():.2%}")

Exercice 1 a completer : entrainer_sgd (retourne None pour l'instant)


In [10]:
# Exercice 2 — Le banc desequilibre, et ce que la precision globale cache
#
# TODO etudiant :
#   1. construire un banc desequilibre (par exemple 120 documents d'une classe contre 20
#      par classe ailleurs) en reutilisant le generateur de la section 1 ;
#   2. reentrainer le modele sur ce banc ;
#   3. implementer precision_par_classe ci-dessous et l'appliquer.
#
# Indice : la matrice de confusion de la section 4 se generalise directement en
#          precision par classe ; inutile de reecrire un second calcul.
#
# Question a trancher a l'ecrit : sur ce banc, la precision globale suffit-elle a decrire
# le comportement du modele ? Repondre par la mesure, pas par l'intuition.

def precision_par_classe(y_vrai, y_pred, categories):
    """Rend un dict {categorie: precision}, la precision etant rappel = justes / support."""
    # TODO etudiant
    return None


resultat = precision_par_classe(y_test, pred_test, CATEGORIES)
if resultat is None:
    print("Exercice 2 a completer : precision_par_classe (retourne None pour l'instant)")
else:
    for categorie, valeur in resultat.items():
        print(f"  {categorie:8} {valeur:.2%}")

Exercice 2 a completer : precision_par_classe (retourne None pour l'instant)


In [11]:
# Exercice 3 — Bigrammes, et le retour de la regularisation
#
# La section 5 a mesure que L2 ne sert a rien sur ce banc : 138 caracteristiques pour
# 144 documents, un rapport de 1 pour 1 et des poids moderes. Le raisonnement y voit la
# condition qui manque pour que la regularisation compte.
#
# TODO etudiant :
#   1. implementer bigrammes(documents) ci-dessous : rendre, pour chaque document, la liste
#      de ses mots SUIVIS de ses paires de mots consecutifs (par exemple "le_match") ;
#   2. construire la matrice sur ce nouveau vocabulaire, en reutilisant construire_matrice ;
#   3. reentrainer et rebalayer lambda (0, 1e-4, 1e-3, 1e-2, 1e-1).
#
# Indice 1 : le vocabulaire passe de ~138 a plusieurs milliers d'entrees, donc
#            caracteristiques >> documents. C'est la condition du regime sur-parametre.
# Indice 2 : la normalisation en frequences relatives reste valable telle quelle.
# Indice 3 : comparer la FORME du balayage a celle de la section 5, pas seulement le
#            meilleur chiffre — c'est la question posee.

def bigrammes(docs):
    """Rend une liste de listes de tokens : les mots, puis les paires consecutives."""
    # TODO etudiant
    return None


jetons = bigrammes(documents)
if jetons is None:
    print("Exercice 3 a completer : bigrammes (retourne None pour l'instant)")
else:
    vocab_bi = sorted({m for doc in jetons for m in doc})
    print(f"vocabulaire unigrammes : {len(vocabulaire)}")
    print(f"vocabulaire bigrammes  : {len(vocab_bi)}  (soit {len(vocab_bi) / len(vocabulaire):.1f}x)")
    print(f"documents d'apprentissage : {len(idx_train)}")
    print(f"rapport caracteristiques / documents : {len(vocab_bi) / len(idx_train):.1f} pour 1")

Exercice 3 a completer : bigrammes (retourne None pour l'instant)


## Références

- D. Jurafsky et J. H. Martin, *Speech and Language Processing*, 3e édition (draft), chapitre « Logistic Regression » — la dérivation de la softmax, de l'entropie croisée et de son gradient y est traitée sous l'angle du TAL.
- C. D. Manning, P. Raghavan et H. Schütze, *Introduction to Information Retrieval*, Cambridge University Press, 2008, chapitre 13 « Text classification and Naive Bayes » — le cadre génératif/discriminatif dont ce carnet et le carnet 10 occupent les deux pôles.
- `scikit-learn`, documentation de `sklearn.linear_model.LogisticRegression` — conventions de régularisation (`C`, multi-classe), utilisées ici comme juge extérieur.

## Navigation

- [<< Précédent — 06. Recherche d'information (TF-IDF, BM25)](06-Recherche-Information-TFIDF-BM25.ipynb) — le carnet 10 occupe le créneau génératif de la classification, et ce carnet 11 le créneau discriminatif. Ce lien pointe le dernier carnet de la série effectivement livré sur `main` ; les carnets 07 à 10 sont annoncés dans le tableau de positionnement mais pas encore publiés, et le renvoi sera ré-aligné quand ils le seront.
- Carnet suivant : à définir lors de la consolidation de la série.